# Lab 04 — Final Validation & Completion Report

This notebook is the final, read-only acceptance gate for Lab 4.

It verifies:
- Bronze and Silver integrity;
- quality/quarantine reconciliation;
- SCD Type 1 and Type 2 invariants;
- version-controlled contract v1/v2 governance;
- controlled schema evolution;
- Delta column mapping;
- Liquid Clustering and maintenance evidence.

Run `lab04_00_setup` manually when structure is first created, then run the production Job tasks `01` through `11`. This notebook performs no reset, merge, optimize, delete, or structural DDL.


## 1. Load the DDL-free runtime configuration

`lab04_00_config` supplies runtime parameters, paths, table names, and the loaded v1/v2 YAML contracts. Structural setup is intentionally separate from the production Job.


In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import BooleanType, StringType, StructField, StructType

objects = {
    "bronze": table_names["bronze"],
    "silver_transactions": table_names["silver_transactions"],
    "quarantine": table_names["quarantine"],
    "quality_metrics": table_names["quality_metrics"],
    "product_scd1": table_names["product_scd1"],
    "product_scd2": table_names["product_scd2"],
    "schema_enforcement_demo": table_names["schema_demo"],
    "schema_evolution_demo": f"{catalog}.{schema}.lab04_schema_evolution_demo",
    "column_mapping_demo": table_names["column_mapping_demo"],
    "maintenance_demo": f"{catalog}.{schema}.lab04_maintenance_demo",
}

validation_rows = []

def table_exists(table_name):
    return spark.catalog.tableExists(table_name)

def add_check(category, rule, expected, actual, passed, severity="ERROR"):
    validation_rows.append({
        "category": str(category),
        "rule": str(rule),
        "expected": str(expected),
        "actual": str(actual),
        "passed": bool(passed),
        "severity": str(severity),
    })

print(f"Validation target: {catalog}.{schema}")
print(f"Batch: {batch_id}; contract: {contract_version}")

Validation target: dbr_dev.parvinbadalov
Batch: initial; contract: v1


## 2. Required-object inventory

Every downstream table is checked before its detailed rules. Missing objects are reported without stopping the notebook early, so the final report shows all remaining gaps in one run.

In [0]:
inventory_rows = []
for object_role, table_name in objects.items():
    exists = table_exists(table_name)
    inventory_rows.append((object_role, table_name, exists))
    add_check("inventory", f"{object_role} exists", "true", exists, exists)

inventory_df = spark.createDataFrame(inventory_rows, ["object_role", "table_name", "exists"])
display(inventory_df.orderBy("object_role"))

object_role,table_name,exists
bronze,dbr_dev.parvinbadalov.lab04_bronze_retail,true
column_mapping_demo,dbr_dev.parvinbadalov.lab04_column_mapping_demo,true
maintenance_demo,dbr_dev.parvinbadalov.lab04_maintenance_demo,true
product_scd1,dbr_dev.parvinbadalov.lab04_product_scd1,true
product_scd2,dbr_dev.parvinbadalov.lab04_product_scd2,true
quality_metrics,dbr_dev.parvinbadalov.lab04_quality_metrics,true
quarantine,dbr_dev.parvinbadalov.lab04_quarantine,true
schema_enforcement_demo,dbr_dev.parvinbadalov.lab04_schema_demo,true
schema_evolution_demo,dbr_dev.parvinbadalov.lab04_schema_evolution_demo,true
silver_transactions,dbr_dev.parvinbadalov.lab04_silver_transactions,true


## 3. Bronze and Silver validation

Bronze must preserve stable source identity and ingestion metadata. Silver must be unique by transaction line, contain the explicit analytics schema, satisfy core business-quality rules, and retain lineage metadata.

In [0]:
bronze_table = objects["bronze"]
if table_exists(bronze_table):
    bronze_df = spark.table(bronze_table)
    bronze_count = bronze_df.count()
    bronze_required = {"_bronze_record_id", "_record_hash", "_batch_id", "_source_file", "_input_file_path", "_bronze_ingested_at"}
    missing = sorted(bronze_required - set(bronze_df.columns))
    add_check("bronze", "contains rows", "> 0", bronze_count, bronze_count > 0)
    add_check("bronze", "required lineage columns", "none missing", missing or "none", not missing)
    if "_bronze_record_id" in bronze_df.columns:
        null_ids = bronze_df.filter(F.col("_bronze_record_id").isNull()).count()
        distinct_ids = bronze_df.select("_bronze_record_id").distinct().count()
        add_check("bronze", "null record IDs", 0, null_ids, null_ids == 0)
        add_check("bronze", "record ID uniqueness", bronze_count, distinct_ids, distinct_ids == bronze_count)

silver_table = objects["silver_transactions"]
if table_exists(silver_table):
    silver_df = spark.table(silver_table)
    silver_count = silver_df.count()
    required_silver = {
        "transaction_line_id", "invoice_no", "stock_code", "description", "quantity",
        "invoice_timestamp", "unit_price", "customer_id", "country", "sales_amount",
        "source_batch_id", "source_file", "input_file_path", "bronze_ingested_at",
        "quality_contract_version", "quality_checked_at", "silver_prepared_at",
    }
    missing = sorted(required_silver - set(silver_df.columns))
    add_check("silver", "contains rows", "> 0", silver_count, silver_count > 0)
    add_check("silver", "explicit analytics schema", "none missing", missing or "none", not missing)
    if "transaction_line_id" in silver_df.columns:
        null_keys = silver_df.filter(F.col("transaction_line_id").isNull()).count()
        distinct_keys = silver_df.select("transaction_line_id").distinct().count()
        add_check("silver", "null transaction keys", 0, null_keys, null_keys == 0)
        add_check("silver", "transaction-key uniqueness", silver_count, distinct_keys, distinct_keys == silver_count)
    business_columns = [c for c in ["invoice_no", "stock_code", "quantity", "unit_price", "invoice_timestamp", "customer_id", "country"] if c in silver_df.columns]
    if business_columns:
        null_business = silver_df.filter(F.expr(" OR ".join([f"`{c}` IS NULL" for c in business_columns]))).count()
        add_check("silver", "required business values", 0, null_business, null_business == 0)
    if {"quantity", "unit_price"}.issubset(silver_df.columns):
        invalid_values = silver_df.filter((F.col("quantity") <= 0) | (F.col("unit_price") <= 0)).count()
        add_check("silver", "positive quantity and price", 0, invalid_values, invalid_values == 0)

## 4. Quarantine and quality-metrics validation

Rejected data must remain auditable, and the metrics table must be idempotent by `(batch_id, contract_version)` with internally reconciled row counts.

In [0]:
quarantine_table = objects["quarantine"]
if table_exists(quarantine_table):
    quarantine_count = spark.table(quarantine_table).count()
    add_check("quality", "quarantine contains rejected rows", "> 0", quarantine_count, quarantine_count > 0)

metrics_table = objects["quality_metrics"]
if table_exists(metrics_table):
    metrics_df = spark.table(metrics_table)
    duplicate_metric_keys = metrics_df.groupBy("batch_id", "contract_version").count().filter(F.col("count") > 1).count()
    reconciliation_failures = metrics_df.filter(F.col("bronze_rows") != F.col("valid_rows") + F.col("rejected_rows")).count()
    percentage_failures = metrics_df.filter(F.abs(F.col("valid_percentage") + F.col("rejected_percentage") - F.lit(100.0)) > F.lit(0.01)).count()
    current_metric_rows = metrics_df.filter((F.col("batch_id") == batch_id) & (F.col("contract_version") == contract_version)).count()
    add_check("quality", "metric-key idempotency", 0, duplicate_metric_keys, duplicate_metric_keys == 0)
    add_check("quality", "metric row reconciliation", 0, reconciliation_failures, reconciliation_failures == 0)
    add_check("quality", "quality percentages total 100", 0, percentage_failures, percentage_failures == 0)
    add_check("quality", "current batch metric row", 1, current_metric_rows, current_metric_rows == 1)

## 5. SCD Type 1 and Type 2 validation

Type 1 must keep one current row per product. Type 2 must keep one current version plus valid, non-overlapping history with unique version numbers.

In [0]:
scd1_keys = None
scd1_table = objects["product_scd1"]
if table_exists(scd1_table):
    scd1_df = spark.table(scd1_table)
    scd1_rows = scd1_df.count()
    scd1_keys = scd1_df.select("stock_code").distinct().count()
    scd1_not_current = scd1_df.filter(~F.col("is_current") | F.col("effective_to").isNotNull()).count()
    add_check("scd1", "one row per product", scd1_keys, scd1_rows, scd1_rows == scd1_keys)
    add_check("scd1", "all rows are current", 0, scd1_not_current, scd1_not_current == 0)

scd2_table = objects["product_scd2"]
if table_exists(scd2_table):
    scd2_df = spark.table(scd2_table)
    scd2_keys = scd2_df.select("stock_code").distinct().count()
    current_key_violations = scd2_df.groupBy("stock_code").agg(F.sum(F.col("is_current").cast("int")).alias("current_rows")).filter(F.col("current_rows") != 1).count()
    duplicate_versions = scd2_df.groupBy("stock_code", "version_number").count().filter(F.col("count") > 1).count()
    invalid_current_end = scd2_df.filter(F.col("is_current") & F.col("effective_to").isNotNull()).count()
    invalid_history_end = scd2_df.filter((~F.col("is_current")) & (F.col("effective_to").isNull() | (F.col("effective_to") <= F.col("effective_from")))).count()
    history_rows = scd2_df.filter(~F.col("is_current")).count()
    temporal_window = Window.partitionBy("stock_code").orderBy("effective_from", "version_number")
    boundary_violations = scd2_df.withColumn("_next_start", F.lead("effective_from").over(temporal_window)).filter(F.col("_next_start").isNotNull() & (F.col("effective_to") != F.col("_next_start"))).count()
    add_check("scd2", "one current row per product", 0, current_key_violations, current_key_violations == 0)
    add_check("scd2", "unique version numbers", 0, duplicate_versions, duplicate_versions == 0)
    add_check("scd2", "current rows have open end", 0, invalid_current_end, invalid_current_end == 0)
    add_check("scd2", "historical rows have valid end", 0, invalid_history_end, invalid_history_end == 0)
    add_check("scd2", "continuous temporal boundaries", 0, boundary_violations, boundary_violations == 0)
    add_check("scd2", "history retained", "> 0", history_rows, history_rows > 0)
    if scd1_keys is not None:
        add_check("scd comparison", "business-key coverage", scd1_keys, scd2_keys, scd1_keys == scd2_keys)

## 6. Contract governance, schema controls, and maintenance validation

The final gate verifies both repository contracts and the persistent Delta demonstrations.

Contract checks prove:
- v1 and v2 load correctly;
- v2 supersedes v1;
- v1 defines 8 source columns;
- v2 defines 10 source columns;
- `loyalty_tier` and `sales_channel` are introduced by v2;
- `Quantity` widens from `integer` to `long`;
- the runtime-selected contract matches the YAML loaded by `lab04_00_config`.

The Delta checks then validate enforcement, evolution, column mapping, and maintenance outputs.


In [0]:
# ---------- Repository contract governance ----------
v1_columns = {
    item["name"]: item
    for item in contract_v1["schema"]["columns"]
}

v2_columns = {
    item["name"]: item
    for item in contract_v2["schema"]["columns"]
}

contract_v1_version = int(contract_v1["contract"]["version"])
contract_v2_version = int(contract_v2["contract"]["version"])
contract_v2_supersedes = int(
    contract_v2["contract"].get("supersedes", -1)
)

contract_changes = sorted(
    set(v1_columns) ^ set(v2_columns)
)

type_changes = sorted(
    name
    for name in (set(v1_columns) & set(v2_columns))
    if v1_columns[name]["type"] != v2_columns[name]["type"]
)

add_check(
    "contracts",
    "v1 YAML version",
    1,
    contract_v1_version,
    contract_v1_version == 1,
)

add_check(
    "contracts",
    "v2 YAML version",
    2,
    contract_v2_version,
    contract_v2_version == 2,
)

add_check(
    "contracts",
    "v2 supersedes v1",
    1,
    contract_v2_supersedes,
    contract_v2_supersedes == 1,
)

add_check(
    "contracts",
    "v1 source column count",
    8,
    len(v1_columns),
    len(v1_columns) == 8,
)

add_check(
    "contracts",
    "v2 source column count",
    10,
    len(v2_columns),
    len(v2_columns) == 10,
)

add_check(
    "contracts",
    "v2 added columns",
    "loyalty_tier, sales_channel",
    ", ".join(
        sorted(set(v2_columns) - set(v1_columns))
    ),
    set(v2_columns) - set(v1_columns)
    == {"loyalty_tier", "sales_channel"},
)

quantity_v1_type = v1_columns.get("Quantity", {}).get("type")
quantity_v2_type = v2_columns.get("Quantity", {}).get("type")

add_check(
    "contracts",
    "Quantity controlled widening",
    "integer -> long",
    f"{quantity_v1_type} -> {quantity_v2_type}",
    (
        quantity_v1_type == "integer"
        and quantity_v2_type == "long"
    ),
)

add_check(
    "contracts",
    "runtime selection matches loaded YAML",
    contract_version,
    f"v{active_contract['contract']['version']}",
    contract_version
    == f"v{active_contract['contract']['version']}",
)

# ---------- Schema enforcement ----------
schema_demo = objects["schema_enforcement_demo"]

if table_exists(schema_demo):
    schema_demo_count = spark.table(schema_demo).count()

    add_check(
        "schema enforcement",
        "strict target contains baseline rows",
        "> 0",
        schema_demo_count,
        schema_demo_count > 0,
    )

# ---------- Schema evolution ----------
evolution_table = objects["schema_evolution_demo"]

if table_exists(evolution_table):
    evolution_df = spark.table(evolution_table)

    evolution_types = {
        field.name: field.dataType.simpleString()
        for field in evolution_df.schema.fields
    }

    evolution_count = evolution_df.count()

    evolution_distinct = (
        evolution_df
        .select("transaction_line_id")
        .distinct()
        .count()
        if "transaction_line_id" in evolution_df.columns
        else -1
    )

    approved_columns = {
        "loyalty_tier",
        "sales_channel",
    }

    missing_evolved = sorted(
        approved_columns - set(evolution_df.columns)
    )

    quantity_type = evolution_types.get("quantity")

    add_check(
        "schema evolution",
        "approved columns added",
        "none missing",
        missing_evolved or "none",
        not missing_evolved,
    )

    add_check(
        "schema evolution",
        "quantity widened",
        "bigint",
        quantity_type,
        quantity_type in {"bigint", "long"},
    )

    add_check(
        "schema evolution",
        "expected demo rows",
        16,
        evolution_count,
        evolution_count == 16,
    )

    add_check(
        "schema evolution",
        "idempotent unique keys",
        evolution_count,
        evolution_distinct,
        evolution_count == evolution_distinct,
    )

# ---------- Column mapping ----------
mapping_table = objects["column_mapping_demo"]

if table_exists(mapping_table):
    mapping_df = spark.table(mapping_table)
    mapping_columns = set(mapping_df.columns)
    mapping_count = mapping_df.count()

    mapping_distinct = (
        mapping_df
        .select("transaction_line_id")
        .distinct()
        .count()
        if "transaction_line_id" in mapping_columns
        else -1
    )

    mapping_detail = (
        spark.sql(f"DESCRIBE DETAIL {mapping_table}")
        .first()
        .asDict(recursive=True)
    )

    mapping_mode = (
        mapping_detail.get("properties") or {}
    ).get("delta.columnMapping.mode")

    add_check(
        "column mapping",
        "mapping mode",
        "name",
        mapping_mode,
        mapping_mode == "name",
    )

    add_check(
        "column mapping",
        "billing_country exists",
        True,
        "billing_country" in mapping_columns,
        "billing_country" in mapping_columns,
    )

    add_check(
        "column mapping",
        "old and dropped columns absent",
        "country, legacy_note absent",
        sorted(
            {"country", "legacy_note"} & mapping_columns
        )
        or "none",
        not (
            {"country", "legacy_note"} & mapping_columns
        ),
    )

    add_check(
        "column mapping",
        "row-key uniqueness preserved",
        mapping_count,
        mapping_distinct,
        mapping_count == mapping_distinct,
    )

# ---------- Maintenance ----------
maintenance_table = objects["maintenance_demo"]

if table_exists(maintenance_table):
    maintenance_detail = (
        spark.sql(f"DESCRIBE DETAIL {maintenance_table}")
        .first()
        .asDict(recursive=True)
    )

    clustering_columns = (
        maintenance_detail.get("clusteringColumns")
        or []
    )

    partition_columns = (
        maintenance_detail.get("partitionColumns")
        or []
    )

    maintenance_count = spark.table(
        maintenance_table
    ).count()

    optimize_runs = (
        spark.sql(
            f"DESCRIBE HISTORY {maintenance_table}"
        )
        .filter(
            F.upper(F.col("operation"))
            == "OPTIMIZE"
        )
        .count()
    )

    add_check(
        "maintenance",
        "table contains rows",
        "> 0",
        maintenance_count,
        maintenance_count > 0,
    )

    add_check(
        "maintenance",
        "Liquid Clustering columns",
        "invoice_date, country",
        sorted(clustering_columns),
        set(clustering_columns)
        == {"invoice_date", "country"},
    )

    add_check(
        "maintenance",
        "classic partitions absent",
        "none",
        partition_columns or "none",
        not partition_columns,
    )

    add_check(
        "maintenance",
        "OPTIMIZE recorded",
        "> 0",
        optimize_runs,
        optimize_runs > 0,
    )


## 7. Consolidated acceptance report

All checks are displayed before the final assertion. This makes troubleshooting easier: one run identifies every failed requirement instead of stopping at the first failure.

In [0]:
validation_schema = StructType([
    StructField("category", StringType(), False),
    StructField("rule", StringType(), False),
    StructField("expected", StringType(), False),
    StructField("actual", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("severity", StringType(), False),
])

validation_df = spark.createDataFrame(validation_rows, validation_schema)
summary_df = (
    validation_df
    .groupBy("category")
    .agg(
        F.count("*").alias("checks"),
        F.sum(F.col("passed").cast("int")).alias("passed"),
        F.sum((~F.col("passed")).cast("int")).alias("failed"),
    )
    .orderBy("category")
)

display(summary_df)
display(validation_df.orderBy(F.col("passed").asc(), "category", "rule"))

required_failures_df = validation_df.filter((F.col("severity") == "ERROR") & (~F.col("passed")))
required_failures = required_failures_df.count()
total_checks = validation_df.count()
passed_checks = validation_df.filter(F.col("passed")).count()

print(f"Lab 4 validation score: {passed_checks}/{total_checks}")
print(f"Required failures: {required_failures}")

if required_failures:
    display(required_failures_df.orderBy("category", "rule"))

if run_validation and required_failures:
    raise AssertionError(f"Lab 4 final validation failed: {required_failures} required checks did not pass.")

print("✅ LAB 4 COMPLETE — all required validation checks passed.")

category,checks,passed,failed
bronze,4,4,0
column mapping,4,4,0
contracts,8,8,0
inventory,10,10,0
maintenance,4,4,0
quality,5,5,0
scd comparison,1,1,0
scd1,2,2,0
scd2,6,6,0
schema enforcement,1,1,0


category,rule,expected,actual,passed,severity
bronze,contains rows,> 0,433737,true,ERROR
bronze,null record IDs,0,0,true,ERROR
bronze,record ID uniqueness,433737,433737,true,ERROR
bronze,required lineage columns,none missing,none,true,ERROR
column mapping,billing_country exists,True,True,true,ERROR
column mapping,mapping mode,name,name,true,ERROR
column mapping,old and dropped columns absent,"country, legacy_note absent",none,true,ERROR
column mapping,row-key uniqueness preserved,200,200,true,ERROR
contracts,Quantity controlled widening,integer -> long,integer -> long,true,ERROR
contracts,runtime selection matches loaded YAML,v1,v1,true,ERROR


Lab 4 validation score: 55/55
Required failures: 0
✅ LAB 4 COMPLETE — all required validation checks passed.


## 8. Evidence and handoff

Capture the following for the README: the object inventory, consolidated category summary, detailed all-green validation table, Silver sample and quality metrics, SCD Type 1/2 comparison, schema-evolution result, column-mapping properties, Liquid Clustering metadata, and the successful Job run.

After validation succeeds:

1. Update `README.md` with architecture, contracts, quality rules, MERGE behavior, SCD trade-offs, schema enforcement/evolution, maintenance, idempotency, and results.
2. Add the production notebook chain to the repository-level Asset Bundle Job; keep the schedule paused during development.
3. Add screenshots under `images/`.
4. Run tests, scan for secrets, commit, and push.

This is the final Lab 4 notebook.